In [67]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA

# Tree-based models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor

# Neural Network
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

# Hyperparameter optimization
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
import warnings
warnings.filterwarnings('ignore')

In [68]:
df = pd.read_csv('../data/gurgaon_properties_post_feature_selection_v2.csv')

In [69]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


In [70]:
df['furnishing_type'].value_counts()

furnishing_type
0.0    2349
1.0    1018
2.0     187
Name: count, dtype: int64

In [71]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [72]:
X = df.drop(columns=['price'])
y = df['price']

In [73]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

### Ordinal Encoding

In [74]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [75]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), [ 'bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ],
    remainder='passthrough'
)

In [76]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [77]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [78]:
scores.mean(),scores.std()

(np.float64(0.7363096633436828), np.float64(0.03238005754429936))

- The results are not very good because we are apply Ordinal Encoding and using Linear Model

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [80]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [81]:
y_pred = pipeline.predict(X_test)

In [82]:
y_pred = np.expm1(y_pred)
mean_absolute_error(np.expm1(y_test),y_pred)

0.9463822160089356

In [83]:
def scorer(model_name, model):
    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    k_fold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores =cross_val_score(pipeline, X, y_transformed, cv = k_fold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [84]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [85]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [86]:
model_output

[['linear_reg', np.float64(0.7363096633436828), 0.9463822160089356],
 ['svr', np.float64(0.7642012011196353), 0.8472636473483922],
 ['ridge', np.float64(0.7363125343993554), 0.9463387741853386],
 ['LASSO', np.float64(0.05943378064493573), 1.528905986892753],
 ['decision tree', np.float64(0.7773372847013122), 0.7418459073210174],
 ['random forest', np.float64(0.8810244191068735), 0.5324854499169372],
 ['extra trees', np.float64(0.8686733424844751), 0.552043527958458],
 ['gradient boosting', np.float64(0.872568092723687), 0.575709755264931],
 ['adaboost', np.float64(0.7525404909316588), 0.7901723672442097],
 ['mlp', np.float64(0.8076661965850447), 0.716112462812412],
 ['xgboost', np.float64(0.8894876835260124), 0.5040475141482346]]

In [87]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [88]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.889488,0.504048
5,random forest,0.881024,0.532485
6,extra trees,0.868673,0.552044
7,gradient boosting,0.872568,0.575710
9,mlp,0.807666,0.716112
4,decision tree,0.777337,0.741846
8,adaboost,0.752540,0.790172
1,svr,0.764201,0.847264
2,ridge,0.736313,0.946339
0,linear_reg,0.736310,0.946382


### OneHotEncoding

In [89]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [90]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [91]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [92]:
scores.mean()

np.float64(0.8546094810971422)

In [93]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [94]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first'),
                                                  ['sector', 'agePossession',
                                                   'furnishing_type'])])),
                ('regressor', LinearRegression())])

In [95]:
y_pred = pipeline.predict(X_test)

In [96]:
y_pred = np.expm1(y_pred)

In [97]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6497514315131458

In [98]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [99]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [100]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [101]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [102]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.895260,0.469479
5,random forest,0.889826,0.492271
10,xgboost,0.895850,0.493456
9,mlp,0.874923,0.532170
7,gradient boosting,0.876863,0.568105
0,linear_reg,0.854609,0.649751
2,ridge,0.854739,0.652915
4,decision tree,0.805731,0.701927
8,adaboost,0.750431,0.821032
1,svr,0.769741,0.834124


### OneHotEncoding With PCA

In [103]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [104]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [105]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [106]:
scores.mean()

np.float64(0.06225201431451135)

In [107]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [108]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [109]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [110]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [111]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.761973,0.648562
6,extra trees,0.739829,0.700927
4,decision tree,0.696442,0.761509
10,xgboost,0.622205,0.967581
7,gradient boosting,0.610623,0.987906
8,adaboost,0.293910,1.358325
1,svr,0.218073,1.361163
9,mlp,0.206215,1.407989
2,ridge,0.062252,1.526707
0,linear_reg,0.062252,1.526707


### Target Encoder

In [112]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [113]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [114]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [115]:
scores.mean(),scores.std()

(np.float64(0.8295219182255362), np.float64(0.018384463379122782))

In [116]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [117]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [118]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [119]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [120]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.904798,0.447518
5,random forest,0.900769,0.452488
6,extra trees,0.902399,0.457749
7,gradient boosting,0.889263,0.509304
4,decision tree,0.832216,0.551907
9,mlp,0.850615,0.630817
8,adaboost,0.818353,0.688856
0,linear_reg,0.829522,0.713011
2,ridge,0.829536,0.713523
1,svr,0.782917,0.818851


### XGBoost Hyperparameter Tuning with Hyperopt

In [121]:
# Define hyperparameter space
space = {
    'n_estimators': hp.choice('n_estimators', [100, 200, 300, 500]),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth': hp.choice('max_depth', [3, 4, 5, 6, 7, 8]),
    'min_child_weight': hp.choice('min_child_weight', [1, 3, 5, 7]),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
    'reg_alpha': hp.uniform('reg_alpha', 0, 1),
    'reg_lambda': hp.uniform('reg_lambda', 1, 2)
}

In [124]:
# Objective function to minimize (MAE)
def objective(params):
    model = XGBRegressor(
        n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        min_child_weight=int(params['min_child_weight']),
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        random_state=42,
        n_jobs=-1
    )
    
    # Use your existing pipeline with target encoding
    pipe = Pipeline([
        ('preprocessor', preprocessor),  # Use the existing preprocessor
        ('regressor', model)
    ])
    
    # Cross-validation
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = cross_val_score(pipe, X, y_transformed, cv=kfold, scoring='neg_mean_absolute_error')
    
    return {'loss': -mae_scores.mean(), 'status': STATUS_OK}

In [125]:
# Run hyperparameter optimization
trials = Trials()
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=50,  # Number of trials
            trials=trials)

100%|██████████| 50/50 [01:32<00:00,  1.86s/trial, best loss: 0.10960181439321157]


In [126]:
# Extract best parameters
best_params = {
    'n_estimators': [100, 200, 300, 500][best['n_estimators']],
    'learning_rate': best['learning_rate'],
    'max_depth': [3, 4, 5, 6, 7, 8][best['max_depth']],
    'min_child_weight': [1, 3, 5, 7][best['min_child_weight']],
    'subsample': best['subsample'],
    'colsample_bytree': best['colsample_bytree'],
    'reg_alpha': best['reg_alpha'],
    'reg_lambda': best['reg_lambda']
}

print("Best Parameters:")
for key, value in best_params.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

# Best MAE score
best_mae = min([trial['result']['loss'] for trial in trials.trials])
print(f"\nBest MAE: {best_mae:.4f}")

Best Parameters:
n_estimators: 500
learning_rate: 0.0389
max_depth: 8
min_child_weight: 1
subsample: 0.8158
colsample_bytree: 0.8402
reg_alpha: 0.0285
reg_lambda: 1.6540

Best MAE: 0.1096


In [127]:
# Train final model with best parameters
final_model = XGBRegressor(
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    min_child_weight=best_params['min_child_weight'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda'],
    random_state=42,
    n_jobs=-1
)

# Final pipeline
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),  # Use the existing preprocessor
    ('regressor', final_model)
])

# Evaluate final model
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
final_scores = cross_val_score(final_pipeline, X, y_transformed, cv=kfold, scoring='neg_mean_absolute_error')
final_r2_scores = cross_val_score(final_pipeline, X, y_transformed, cv=kfold, scoring='r2')

print(f"Final Tuned Model Results:")
print(f"MAE: {-final_scores.mean():.4f} (+/- {final_scores.std()*2:.4f})")
print(f"R²: {final_r2_scores.mean():.4f} (+/- {final_r2_scores.std()*2:.4f})")

Final Tuned Model Results:
MAE: 0.1078 (+/- 0.0146)
R²: 0.9087 (+/- 0.0272)


In [129]:
# Save the final pipeline and input data
import pickle

# Train the final pipeline on full dataset
final_pipeline.fit(X, y_transformed)

# Save the trained pipeline
with open('../models/final_xgb_pipeline.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)

# Save the input data (X)
with open('../models/input_data_X.pkl', 'wb') as f:
    pickle.dump(X, f)
